In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
DATASET_DIR = Path("/content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3")

METADATA_CSV = Path(
    "/content/drive/MyDrive/Speciale/dataset/prepared_overhead_rgb_dataset/"
    "dish_carb_split_metadata.csv"
)

OUTPUT_DIR = Path("/content/drive/MyDrive/Speciale/Models/simple_mean_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BAD_DISH_IDS = {
    "dish_1559172356",
    "dish_1564588719",
}

print("METADATA_CSV exists:", METADATA_CSV.exists())
print("DATASET_DIR exists:", DATASET_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

METADATA_CSV exists: True
DATASET_DIR exists: True
OUTPUT_DIR: /content/drive/MyDrive/Speciale/Models/simple_mean_baseline


In [ ]:
df = pd.read_csv(METADATA_CSV)

print("Loaded metadata shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Loaded metadata shape: (3262, 4)
Columns:
['dish_id', 'total_carb', 'split', 'source_cafe']


,dish_id,total_carb,split,source_cafe
0,dish_1556575327,10.618,test,cafe1
1,dish_1557861216,0.000,test,cafe1
2,dish_1557862345,0.000,test,cafe1
3,dish_1557862696,3.850,test,cafe1
4,dish_1557862738,0.000,test,cafe1


In [ ]:
required_cols = ["dish_id", "total_carb", "split"]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["dish_id"] = df["dish_id"].astype(str).str.strip()
df["total_carb"] = pd.to_numeric(df["total_carb"], errors="coerce")

print("Rows before removing bad dishes:", len(df))
print("Unique dishes before removing bad dishes:", df["dish_id"].nunique())

df = df[~df["dish_id"].isin(BAD_DISH_IDS)].copy()

print("Rows after removing bad dishes:", len(df))
print("Unique dishes after removing bad dishes:", df["dish_id"].nunique())
print("Removed bad dishes:", BAD_DISH_IDS)

Rows before removing bad dishes: 3262
Unique dishes before removing bad dishes: 3262
Rows after removing bad dishes: 3260
Unique dishes after removing bad dishes: 3260
Removed bad dishes: {'dish_1564588719', 'dish_1559172356'}


In [ ]:
df["rgb_path"] = df["dish_id"].apply(
    lambda d: str(DATASET_DIR / str(d) / "overhead" / "rgb.png")
)

df["rgb_exists"] = df["rgb_path"].apply(lambda p: Path(p).exists())

def file_size_or_minus1(path_str):
    p = Path(path_str)
    try:
        return p.stat().st_size
    except Exception:
        return -1

df["rgb_size"] = df["rgb_path"].apply(file_size_or_minus1)

print("Missing RGB images:", (~df["rgb_exists"]).sum())
print("Empty/unreadable RGB images:", (df["rgb_size"] <= 0).sum())

df = df[
    (df["rgb_exists"]) &
    (df["rgb_size"] > 0) &
    (df["total_carb"].notna())
].copy()

print("\nRemaining rows after filtering:")
print(df["split"].value_counts(dropna=False))
print("Total remaining:", len(df))

Missing RGB images: 0
Empty/unreadable RGB images: 0

Remaining rows after filtering:
split
train    2419
test      507
val       334
Name: count, dtype: int64
Total remaining: 3260


In [ ]:
train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
test_df  = df[df["split"] == "test"].copy().reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows: ", len(test_df))

if len(train_df) == 0:
    raise ValueError("Train split is empty.")

if len(test_df) == 0:
    raise ValueError("Test split is empty.")

Train rows: 2419
Test rows:  507


In [ ]:
train_mean_carb = train_df["total_carb"].mean()

print(f"Mean carbohydrate value in training set: {train_mean_carb:.4f} g")

test_eval_df = test_df.copy()
test_eval_df["prediction"] = train_mean_carb

test_eval_df["signed_error"] = test_eval_df["prediction"] - test_eval_df["total_carb"]
test_eval_df["abs_error"] = np.abs(test_eval_df["signed_error"])
test_eval_df["squared_error"] = test_eval_df["signed_error"] ** 2

display(test_eval_df.head())

Mean carbohydrate value in training set: 19.4157 g


,dish_id,total_carb,split,source_cafe,rgb_path,rgb_exists,rgb_size,prediction,signed_error,abs_error,squared_error
0,dish_1556575327,10.618,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,548259,19.415721,8.797721,8.797721,77.399901
1,dish_1557861216,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,353804,19.415721,19.415721,19.415721,376.970236
2,dish_1557862345,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,407501,19.415721,19.415721,19.415721,376.970236
3,dish_1557862696,3.850,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,391780,19.415721,15.565721,15.565721,242.291681
4,dish_1557862738,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,412017,19.415721,19.415721,19.415721,376.970236


In [ ]:
def compute_pmae(y_true, y_pred):
    """
    Nutrition5K-style percentage mean absolute error:
    PMAE = 100 * MAE / mean(y_true)
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    mae = np.mean(np.abs(y_pred - y_true))
    mean_true = np.mean(y_true)

    return 100 * mae / mean_true


y_true = test_eval_df["total_carb"].values
y_pred = test_eval_df["prediction"].values

mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mse)
pmae = compute_pmae(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

baseline_metrics = {
    "baseline": "training_mean_prediction",
    "train_mean_carb": float(train_mean_carb),
    "test_mean_carb": float(np.mean(y_true)),
    "mean_prediction": float(np.mean(y_pred)),
    "mse": float(mse),
    "mae": float(mae),
    "rmse": float(rmse),
    "pmae": float(pmae),
    "r2": float(r2),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}

print(json.dumps(baseline_metrics, indent=2))

{
  "baseline": "training_mean_prediction",
  "train_mean_carb": 19.415721350971474,
  "test_mean_carb": 19.683595331360944,
  "mean_prediction": 19.415721350971474,
  "mse": 303.6001493866208,
  "mae": 13.733366833401417,
  "rmse": 17.424125498475405,
  "pmae": 69.77062225341797,
  "r2": -0.00023640776627198434,
  "n_train": 2419,
  "n_test": 507
}


In [ ]:
baseline_predictions_path = OUTPUT_DIR / "simple_train_mean_baseline_test_predictions.csv"
baseline_summary_path = OUTPUT_DIR / "simple_train_mean_baseline_summary.json"

test_eval_df.to_csv(baseline_predictions_path, index=False)

with open(baseline_summary_path, "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print("Saved predictions to:", baseline_predictions_path)
print("Saved summary to:", baseline_summary_path)

Saved predictions to: /content/drive/MyDrive/Speciale/Models/simple_mean_baseline/simple_train_mean_baseline_test_predictions.csv
Saved summary to: /content/drive/MyDrive/Speciale/Models/simple_mean_baseline/simple_train_mean_baseline_summary.json


In [ ]:
latex_row = (
    f"Training mean baseline & "
    f"{baseline_metrics['mse']:.4f} & "
    f"{baseline_metrics['mae']:.4f} & "
    f"{baseline_metrics['rmse']:.4f} & "
    f"{baseline_metrics['pmae']:.2f}\\% \\\\"
)

print(latex_row)

Training mean baseline & 303.6001 & 13.7334 & 17.4241 & 69.77\% \\


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

train_mean_carb = train_df["total_carb"].mean()

test_eval_df = test_df.copy()
test_eval_df["prediction"] = train_mean_carb

y_true = test_eval_df["total_carb"].values
y_pred = test_eval_df["prediction"].values

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)

metrics_table = pd.DataFrame({
    "Model": ["Simple training mean baseline"],
    "MSE": [mse],
    "RMSE": [rmse],
    "MAE": [mae]
})

display(metrics_table.round(4))

,Model,MSE,RMSE,MAE
0,Simple training mean baseline,303.6001,17.4241,13.7334
